# 02 · Data visualization

This shared EDA notebook examines class balance, feature distributions by diagnosis, and relationships among representative measurements. It uses the raw dataset without modifying it. The all-empty `Unnamed: 32` column is excluded from analyses; `id` is never treated as a predictor.

**Interpretation guardrail:** plots are exploratory. Separation in this dataset does not demonstrate clinical performance or generalization. No feature will be dropped or transformed here based only on appearance. There is no temporal field in the supplied table, so temporal plots are not applicable.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_columns', 40)
sns.set_theme(style='whitegrid', context='notebook', palette=['#0072B2', '#D55E00'])

DATA_PATH = Path('../data/raw/Raw_Breast_Cancer_Data.csv')
TARGET = 'diagnosis'
ID_COLUMN = 'id'
CLASS_ORDER = ['B', 'M']
COLORS = {'B': '#0072B2', 'M': '#D55E00'}
df_raw = pd.read_csv(DATA_PATH)
empty_columns = df_raw.columns[df_raw.isna().all()].tolist()
df = df_raw.drop(columns=empty_columns).copy()
features = df.drop(columns=[TARGET, ID_COLUMN])
print(f'Rows: {len(df):,}; numeric measurements: {features.shape[1]}')
print(f'Excluded all-empty columns: {empty_columns}')
print(f'Target labels: {sorted(df[TARGET].dropna().unique())}')

## 1. How many records belong to each diagnosis?

The first chart shows counts and within-dataset percentages. Use the class counts to keep later validation stratified and to interpret recall/precision alongside any aggregate score.

In [ ]:
counts = df[TARGET].value_counts().reindex(CLASS_ORDER)
percentages = counts / counts.sum() * 100
fig, ax = plt.subplots(figsize=(7.5, 4.8), constrained_layout=True)
bars = ax.bar(["Benign (B)", "Malignant (M)"], counts.values, color=[COLORS[c] for c in CLASS_ORDER], width=0.62)
ax.set_title('Diagnosis counts in the supplied dataset', loc='left', weight='bold', pad=12)
ax.set_ylabel('Records')
ax.set_xlabel('Diagnosis')
ax.spines[['top', 'right']].set_visible(False)
ax.bar_label(bars, labels=[f'{v:,}  ({p:.1f}%)' for v, p in zip(counts.values, percentages.values)], padding=4)
ax.set_ylim(0, counts.max() * 1.18)
plt.show()
display(pd.DataFrame({'count': counts, 'percent': percentages.round(1)}))

## 2. How do selected measurements vary by diagnosis?

Overlaid histograms with density scaling compare representative mean measurements without implying that these four features are the full modeling set. Evaluate overlap as well as differences; scaling and correlated predictors will be handled inside model pipelines later.

In [ ]:
distribution_features = ['radius_mean', 'texture_mean', 'area_mean', 'concave points_mean']
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, feature in zip(axes.flat, distribution_features):
    sns.histplot(data=df, x=feature, hue=TARGET, hue_order=CLASS_ORDER,
                 stat='density', common_norm=False, element='step', fill=True,
                 alpha=0.22, palette=COLORS, ax=ax)
    ax.set_title(feature.replace('_', ' ').title(), loc='left', weight='bold')
    ax.set_ylabel('Density')
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Selected measurement distributions by diagnosis', fontsize=15, weight='bold', x=0.02, ha='left')
plt.show()

## 3. Which selected measurements move together?

The correlation heatmap uses the pooled records and is descriptive only. It can reveal redundancy among geometric measurements; it does not establish causality or feature importance. Correlation will inform interpretation, not automatic feature removal.

In [ ]:
correlation_features = [
    'radius_mean', 'perimeter_mean', 'area_mean', 'texture_mean',
    'radius_se', 'perimeter_se', 'area_se',
    'radius_worst', 'perimeter_worst', 'area_worst',
    'compactness_worst', 'concavity_worst', 'concave points_worst',
]
corr = df[correlation_features].corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(11, 9), constrained_layout=True)
sns.heatmap(corr, mask=mask, cmap='vlag', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'label': 'Spearman correlation'}, ax=ax)
ax.set_title('Rank correlations among representative measurements', loc='left', weight='bold', pad=14)
ax.tick_params(axis='x', rotation=50)
ax.tick_params(axis='y', rotation=0)
plt.show()

## 4. Compare spread and outliers across diagnoses

Boxplots summarize medians and spread for four representative measurements. Individual points are omitted to keep the view readable; the distributions above show shape and overlap. Outliers are not automatically errors and should not be removed without a data-quality reason.

In [ ]:
spread_features = ['radius_mean', 'texture_mean', 'area_mean', 'concave points_mean']
plot_data = df.melt(id_vars=TARGET, value_vars=spread_features, var_name='measurement', value_name='value')
fig, ax = plt.subplots(figsize=(12, 5.5), constrained_layout=True)
sns.boxplot(data=plot_data, x='measurement', y='value', hue=TARGET, hue_order=CLASS_ORDER,
            palette=COLORS, showfliers=False, width=0.7, ax=ax)
ax.set_title('Measurement spread by diagnosis (fliers hidden for readability)', loc='left', weight='bold', pad=12)
ax.set_xlabel('Measurement')
ax.set_ylabel('Observed value')
ax.set_xticklabels([s.replace('_', ' ').title() for s in spread_features])
ax.legend(title='Diagnosis', labels=['Benign (B)', 'Malignant (M)'], frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

## Interpretation notes to complete after running

- Record visible class overlap and distribution shape without claiming a validated decision boundary.
- Identify strong correlation patterns shown by the heatmap and carry them forward as a multicollinearity consideration.
- Note any unusual tails or spread, but do not remove observations on visual grounds alone.
- Confirm whether the target counts align with the data overview.

### Self-critique

The strongest limitation is that these plots describe one modest, curated dataset and selected features; visual class separation can make the problem look easier than it will be on genuinely external data. The notebook limits conclusions to exploratory patterns and leaves model selection to leakage-safe validation and a held-out test set.